In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

train_df = pd.read_csv(
    "../data/processed/train_data.csv",
    parse_dates=["snapshot_date"]
)

validation_df = pd.read_csv(
    "../data/processed/validation_data.csv",
    parse_dates=["snapshot_date"]
)

test_df = pd.read_csv(
    "../data/processed/test_data.csv",
    parse_dates=["snapshot_date"]
)

print("Training shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Testing shape:", test_df.shape)

In [ ]:
target_column = "churn"

numeric_columns = [
    "tx_count_90d",
    "total_amount_90d",
    "avg_amount_90d",
    "median_amount_90d",
    "std_amount_90d",
    "min_amount_90d",
    "max_amount_90d",
    "active_days_90d",
    "days_since_last_tx",
    "days_since_first_tx",
    "weekend_tx_ratio",
    "avg_tx_per_active_day",
    "tx_type_deposit_count",
    "tx_type_payment_count",
    "tx_type_transfer_count",
    "tx_type_withdrawal_count",
    "tx_type_deposit_share",
    "tx_type_payment_share",
    "tx_type_transfer_share",
    "tx_type_withdrawal_share",
    "age",
    "household_size"
]

categorical_columns = [
    "gender",
    "location",
    "income_bracket",
    "occupation",
    "education_level",
    "marital_status",
    "acquisition_channel"
]

log_columns = [
    "tx_count_90d",
    "total_amount_90d",
    "avg_amount_90d",
    "median_amount_90d",
    "std_amount_90d",
    "min_amount_90d",
    "max_amount_90d",
    "avg_tx_per_active_day",
    "tx_type_deposit_count",
    "tx_type_payment_count",
    "tx_type_transfer_count",
    "tx_type_withdrawal_count"
]

feature_columns = numeric_columns + categorical_columns

X_train = train_df[feature_columns].copy()
y_train = train_df[target_column].copy()

X_validation = validation_df[feature_columns].copy()
y_validation = validation_df[target_column].copy()

X_test = test_df[feature_columns].copy()
y_test = test_df[target_column].copy()

print("Numerical features:", len(numeric_columns))
print("Categorical features:", len(categorical_columns))
print("Log-transformed features:", len(log_columns))
print("Total predictors:", len(feature_columns))

print("\nMissing columns:")
print(set(feature_columns) - set(train_df.columns))

print("\nMissing training values:", X_train.isnull().sum().sum())
print("Missing validation values:", X_validation.isnull().sum().sum())
print("Missing testing values:", X_test.isnull().sum().sum())

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).mul(100).round(2))

In [ ]:
def create_preprocessor():

    log_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("log_transform", FunctionTransformer(np.log1p)),
        ("scaler", StandardScaler())
    ])

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    remaining_numeric_columns = [
        col for col in numeric_columns
        if col not in log_columns
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            ("log_numeric", log_pipeline, log_columns),
            ("numeric", numeric_pipeline, remaining_numeric_columns),
            ("categorical", categorical_pipeline, categorical_columns)
        ],
        remainder="drop"
    )

    return preprocessor

In [ ]:
def evaluate_model(model, X, y):

    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]

    return {
        "Accuracy": accuracy_score(y, y_pred),
        "Precision": precision_score(y, y_pred, zero_division=0),
        "Recall": recall_score(y, y_pred, zero_division=0),
        "F1 Score": f1_score(y, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y, y_prob),
        "PR-AUC": average_precision_score(y, y_prob)
    }

In [ ]:
models = {
    "Dummy Classifier": DummyClassifier(
        strategy="most_frequent"
    ),

    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=10,
        min_samples_leaf=50,
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=15,
        min_samples_leaf=10,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    )
}

In [ ]:
trained_models = {}
validation_results = []

for model_name, classifier in models.items():

    print(f"Training {model_name}...")

    model_pipeline = Pipeline([
        ("preprocessor", create_preprocessor()),
        ("classifier", classifier)
    ])

    model_pipeline.fit(X_train, y_train)

    trained_models[model_name] = model_pipeline

    if model_name == "Dummy Classifier":
        metrics = {
            "Accuracy": accuracy_score(
                y_validation,
                model_pipeline.predict(X_validation)
            ),
            "Precision": precision_score(
                y_validation,
                model_pipeline.predict(X_validation),
                zero_division=0
            ),
            "Recall": recall_score(
                y_validation,
                model_pipeline.predict(X_validation),
                zero_division=0
            ),
            "F1 Score": f1_score(
                y_validation,
                model_pipeline.predict(X_validation),
                zero_division=0
            ),
            "ROC-AUC": np.nan,
            "PR-AUC": np.nan
        }
    else:
        metrics = evaluate_model(
            model_pipeline,
            X_validation,
            y_validation
        )

    metrics["Model"] = model_name
    validation_results.append(metrics)

validation_results_df = pd.DataFrame(validation_results)

validation_results_df = validation_results_df[
    ["Model", "Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC", "PR-AUC"]
]

display(
    validation_results_df
    .sort_values("PR-AUC", ascending=False)
    .round(4)
)

In [ ]:
selected_model_name = "Logistic Regression"
selected_threshold = 0.50

selected_model = trained_models[selected_model_name]

print("Selected model:", selected_model_name)
print("Classification threshold:", selected_threshold)

In [ ]:
validation_predictions = selected_model.predict(X_validation)

print("Validation Confusion Matrix:")
print(confusion_matrix(y_validation, validation_predictions))

print("\nValidation Classification Report:")
print(
    classification_report(
        y_validation,
        validation_predictions,
        zero_division=0
    )
)

In [ ]:
os.makedirs("../models", exist_ok=True)
os.makedirs("../reports", exist_ok=True)

joblib.dump(
    selected_model,
    "../models/logistic_regression_pipeline.pkl"
)

print("Selected Logistic Regression model saved successfully.")

In [ ]:
validation_results_df.to_csv(
    "../reports/model_validation_results.csv",
    index=False
)

print("Model validation results saved successfully.")

In [ ]:
model_metadata = {
    "selected_model": selected_model_name,
    "classification_threshold": selected_threshold,
    "target": target_column,
    "training_period": "2023-04-30 to 2023-09-30",
    "validation_period": "2023-10-31",
    "test_period": "2023-11-29",
    "number_of_features": len(feature_columns),
    "numerical_features": numeric_columns,
    "categorical_features": categorical_columns,
    "log_transformed_features": log_columns
}

with open("../models/model_metadata.json", "w") as file:
    json.dump(model_metadata, file, indent=4)

print("Model metadata saved successfully.")